## 1) Colab runtime check (GPU)

Run this cell to check CUDA availability and print the GPU name. In Colab, set Runtime → Change runtime type → GPU before running.


## 2) Install dependencies

This installs the main libraries. On Colab the runtime may require a restart after some installs (e.g., bitsandbytes). Run this cell and follow any prompts. Adjust versions if you need specific CUDA/tooling compatibility.


## 3) Authenticate with Hugging Face Hub

If the model requires acceptance of terms or is gated, you must provide a Hugging Face token with access. You can create one at https://huggingface.co/settings/tokens and paste it when prompted. The token will be used by `huggingface_hub.login()` to fetch model files.


## 4) (Optional) Mount Google Drive

Mount Google Drive if you want to persist downloads, converted weights, or generated outputs across sessions.


### 5) Load tokenizer and LLaMA-7B model (transformers)




In [6]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="yahma/llama-7b-hf")

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("yahma/llama-7b-hf")
model = AutoModelForCausalLM.from_pretrained("yahma/llama-7b-hf")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## 7) Simple inference test

Run a short generation to verify that the model responds. Change `prompt` to test other queries.


In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
try:
    dev = next(model.parameters()).device
    print("Model parameter device:", dev)
except Exception as e:
    print("Cannot read model device:", e)

CUDA available: True
Model parameter device: cpu


In [11]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)   # <-- key line
model.eval()

prompt = "Explain the DoRA method for parameter-efficient fine-tuning in one short paragraph."
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)

print(tokenizer.decode(out[0], skip_special_tokens=True))

Explain the DoRA method for parameter-efficient fine-tuning in one short paragraph.

\begin{blockquote}

The DoRA method is a parameter-efficient fine-tuning method that
  uses a small number of parameters to achieve a high degree of
  generalization. It is based on the idea that the parameters of a
  neural network are not independent, and that the parameters of a
  neural network can be grouped into a small number of clusters.
  The DoRA method uses a small number of parameters to represent each
  cluster, and then uses a small number of parameters to represent the
  entire network.
\end{blockquote}

Comment: I


In [12]:
!nvidia-smi

Sat Apr 18 02:08:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   32C    P0             88W /  600W |   26537MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Set the model id or local path here
# Replace with the model you have access to. Examples:
#  - 'decapoda-research/llama-7b-hf'  (community-converted LLaMA checkpoints on HF; requires access/acceptance)
#  - 'meta-llama/Llama-2-7b-chat-hf'  (LLaMA-2 chat model on HF; requires agreement)

MODEL_ID_OR_PATH = "decapoda-research/llama-7b-hf"  # <-- change if needed


## 6) Load tokenizer and LLaMA-7B model (transformers)

This example attempts to load the model using `device_map='auto'` and `torch_dtype=torch.float16`. If you run out of memory, see the next section for 8-bit/bitsandbytes options or use a larger GPU.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load tokenizer
print("Loading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH, use_fast=False)
    print("Tokenizer loaded")
except Exception as e:
code
python
# 1) Install required packages (run once). Restart runtime if bitsandbytes is installed/updated.
!pip install -qU transformers datasets accelerate safetensors huggingface_hub einops
!pip install -qU bitsandbytes || true
print('Installed packages (restart runtime after bitsandbytes if newly installed)')

code
python
# 2) (Optional) Mount Google Drive and copy repo into /content/DoRA if present on Drive.
import os, shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
except Exception as e:
    print('Not running inside Colab or google.colab not available:', e)

if not os.path.exists('/content/DoRA'):
    src = '/content/drive/MyDrive/DoRA'
    if os.path.exists(src):
        print('Copying DoRA from Drive to /content/DoRA (this may take a moment)')
        try:
            shutil.copytree(src, '/content/DoRA')
            print('Copied DoRA to /content/DoRA')
        except Exception as e:
            print('Failed to copy:', e)
    else:
        print('No DoRA folder found in Drive at', src, '. If you uploaded the repo differently, ensure /content/DoRA exists before running the experiment.')
else:
    print('/content/DoRA already exists')

code
python
# 3) (Optional) Login to Hugging Face to access gated models
from getpass import getpass
from huggingface_hub import login
token = getpass('Enter your Hugging Face token (leave blank to skip): ')
if token:
    login(token)
    print('Logged in to Hugging Face')
else:
    print('Continuing anonymously; gated repos may fail to load')

code
python
# 4) Experiment hyperparameters — edit as needed before running the finetune cell
BASE_MODEL = 'yahma/llama-7b-hf'  # change to the HF id or local path you want to use
RANK = 32
ALPHA = 64
DROPOUT = 0.05
LR = 1e-4
BATCH_SIZE = 16
EPOCHS = 3
MAX_TRAIN_SAMPLES = 256
print('Experiment config set. Edit variables and re-run this cell if needed.')

code
python
# 5) Run the unit test for the DoRA adapter (quick sanity check)
import sys, os
# ensure the repository root is on PYTHONPATH
repo_root = '/content/DoRA'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

    from tests.test_dora_adapter import test_forward_and_update
    test_forward_and_update()
    print('DoRA adapter unit test passed')
except Exception as e:
    print('Unit test failed or repository not found at /content/DoRA:', e)
    print('Make sure you copied the DoRA repo into /content/DoRA or edit repo_root variable to point to the project root.')

code
python
# 6) Run the minimal finetune experiment (this will run inside the repo at /content/DoRA)
import subprocess, sys, os
repo_root = '/content/DoRA'
if not os.path.exists(repo_root):
    raise RuntimeError('Repo not found at /content/DoRA. Copy or upload it to Colab before running this cell.')
cmd = [sys.executable, os.path.join(repo_root, 'scripts', 'finetune_dora.py'), '--base_model', BASE_MODEL, '--rank', str(RANK), '--alpha', str(ALPHA), '--dropout', str(DROPOUT), '--batch_size', str(BATCH_SIZE), '--lr', str(LR), '--epochs', str(EPOCHS), '--max_train_samples', str(MAX_TRAIN_SAMPLES)]
print('Running command:', ' '.join(cmd))
res = subprocess.run(cmd)
print('Finetune process exited with code', res.returncode)

    print("Tokenizer load error:", e)
    raise

# Load model
print("Loading model (this may take a few minutes)...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_OR_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()
    print("Model loaded")
except Exception as e:
    print("Model load failed:", e)
    print("If you run out of memory, consider using 8-bit quantization (bitsandbytes) or a larger GPU runtime.")
    raise


## 7) Simple inference test

Run a short generation to verify that the model responds. Change `prompt` to test other queries.


In [ ]:
prompt = "Explain the DoRA method for parameter-efficient fine-tuning in a concise paragraph."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

import time
start = time.time()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
end = time.time()
print("Generation time (s):", round(end-start, 2))
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 8) (Optional) bitsandbytes 8-bit setup for memory-constrained GPUs

If you run out of memory, consider using 8-bit quantization via bitsandbytes. Install `bitsandbytes` (already installed above) and re-run the load with `load_in_8bit=True` and `device_map='auto'`.

Example (uncomment to use):

# from transformers import AutoTokenizer, AutoModelForCausalLM
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH, use_fast=False)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID_OR_PATH,
#     load_in_8bit=True,
#     device_map='auto',
#     trust_remote_code=True
# )


## 9) Save generated output to Drive (optional)

If you mounted Drive earlier, you can save generated outputs so they persist across sessions. Example below writes the last output to Drive.


In [ ]:
try:
    out_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    out_path = '/content/drive/MyDrive/llama_generation_output.txt'
    with open(out_path, 'w') as f:
        f.write(out_text)
    print(f"Saved output to {out_path}")
except Exception as e:
    print('Could not save output (maybe Drive not mounted):', e)


## Notes and troubleshooting

- If you get authentication/permission errors when loading the model from Hugging Face, ensure you have accepted the model license or you have uploaded local converted weights.
- If model loading fails due to memory, try `load_in_8bit=True` (bitsandbytes) or use a larger GPU (e.g., Colab Pro/GPU with more memory).
- If you see `trust_remote_code=True` warnings, they are expected for community-converted models; inspect the remote code before use for security.

Enjoy — this notebook gives you a working starting point to load and test a LLaMA-7B-compatible model in Colab.
